# 04 · Load Box Office

**What this does:** loads the newest US cinema box office delivery (Comscore daily gross per theater and title) from Google Cloud Storage into BigQuery. It is used to work out each film's **opening weekend box office** (release day, Thursday or Friday, through Sunday).

| | |
|---|---|
| **When to run** | After a new box office export from NBC has been saved to `gs://nbcu-marcus-hilliard/up_bo_usa/<YYYYMMDD>/` |
| **How long** | Check only: about 1 minute. Full load: about 5 minutes. |
| **What it changes** | Only the table `us_hment_mmm_raw.box_office_usa`, and only when `DRY_RUN = False` |
| **Safe to re-run?** | Yes. A delivery that is already loaded is skipped. |

### How to use it
1. Leave the settings in **Step 1** as they are and click **Run ▸ Run All Cells**. This is a *check only*; nothing changes.
2. Read the message in **Step 2**. If it says **✅ PASS — ready to load**, set `DRY_RUN = False` and **Run All** again.
3. Look at **Step 3** to confirm the new delivery is in the table.

> Only edit the cell in Step 1. Everything below it runs on its own.

## Step 1 · Settings

In [1]:
# Which delivery folder to load: "latest" (recommended) or a specific date such as "20261008"
DELIVERY = "latest"

# True  = check only, nothing is changed (always start here)
# False = actually load the data
DRY_RUN = False

# Advanced: leave these as False unless told otherwise
FORCE_RELOAD = False              # True = reload even if this delivery was already loaded
ALLOW_SMALLER_DELIVERY = False    # True = load even if it is smaller than the previous delivery

## Step 2 · Run the checks and the load
*Nothing to edit here.* The detailed log prints first, followed by a short result message.

In [2]:
from IPython.display import Markdown, display

from us_hment_mmm.boxoffice.load_raw import run, show_raw_table_summary, show_recent_loads

try:
    result = run(delivery=DELIVERY, dry_run=DRY_RUN, force=FORCE_RELOAD,
                 allow_smaller=ALLOW_SMALLER_DELIVERY)
except Exception as exc:
    display(Markdown(
        f"### 🛑 STOP — the load failed\n\n`{exc}`\n\n"
        "Nothing was published; the existing box office table is unchanged. "
        "Save this notebook and send it to the pipeline owner."))
    raise

status = result["status"]
problems = "\n".join(f"- {p}" for p in result["problems"])
warnings = "\n".join(f"- {w}" for w in result["warnings"])

messages = {
    "READY": f"### ✅ PASS — ready to load\nDelivery **{result['delivery']}** passed every check.\n\n"
             "**Next:** in Step 1 set `DRY_RUN = False`, then **Run All** again.",
    "SUCCESS": f"### ✅ PASS — loaded\nDelivery **{result['delivery']}** is now in BigQuery "
               f"({result.get('rows_loaded', 0):,} rows). See Step 3.",
    "WOULD_SKIP": f"### ℹ️ Already loaded\nDelivery **{result['delivery']}** is already in BigQuery. "
                  "Nothing to do. (To reload it anyway, set `FORCE_RELOAD = True` in Step 1.)",
    "SKIPPED": f"### ℹ️ Already loaded\nDelivery **{result['delivery']}** is already in BigQuery. "
               "Nothing was changed.",
    "WOULD_BLOCK": f"### 🛑 STOP — this delivery cannot be loaded\n{problems}\n\n"
                   "**What to do:** if a delivery is smaller than the previous one, the NBC export was "
                   "probably incomplete; re-export it. For any other problem, contact the pipeline owner.",
    "BLOCKED": f"### 🛑 STOP — not loaded\n{problems}\n\nNothing was changed. "
               "Contact the pipeline owner.",
}
display(Markdown(messages.get(status, f"Status: {status}")))
if warnings:
    display(Markdown(f"**Notes (not blocking):**\n{warnings}"))

BOX OFFICE LOAD  (LOAD)
Delivery folders found:
  20261030:  170 csv files,    8.06 GB

Selected delivery: 20261030   (previous: none)
Reading the header of every file (first 64 KB each)...
Columns: 14, all required columns present, same header in every file.

Step 1/3  Load files into the stage table
  Loading gs://nbcu-marcus-hilliard/up_bo_usa/20261030/*.csv
  -> prj-cgh-glbl-serv-npd-glbl.us_hment_mmm_raw._stage_box_office_usa (server-side; this can take several minutes)
  Rows read from files: 60,922,862
Step 2/3  Replace this delivery in the raw table
  Rows in raw table for 20261030: 60,922,862
Step 3/3  Keep newest 2 deliveries; tidy up

Raw table box_office_usa now holds:
  delivery               rows  first day   last day            amount USD
  2026-10-30       60,922,862  2021-01-01  2026-10-29   44,288,851,983.87

Done.


### ✅ PASS — loaded
Delivery **20261030** is now in BigQuery (60,922,862 rows). See Step 3.

## Step 3 · Check the result
**What's in the box office table now** (one row per delivery kept), and **the last five load attempts**.

In [3]:
display(Markdown("**Box office table contents**"))
display(show_raw_table_summary())
display(Markdown("**Recent load attempts**"))
display(show_recent_loads(5))

**Box office table contents**

Delivery,Rows,First day,Last day,Box office (USD)
2026-10-30,"60,922,862",2021-01-01,2026-10-29,"$44,288,851,984"


**Recent load attempts**

Finished (Central),Delivery,Status,Rows,Files,Size (GB),Message
2026-10-30 22:31,20261030,SUCCESS,"60,922,862",170,8.06,Loaded


---
### Running this as a Domino Job instead
For a long load, or to avoid keeping a workspace open, start a Domino Job with this command:

```
python -m us_hment_mmm.boxoffice.load_raw
```

Add `--dry-run` to check only.